# Model Trainging – Heart Disease Dataset
*Training and evaluation of models for Cardio - Risk Prediction project*  

---

## Table of Contents

### [1. Reproducibility & Imports](#imports)
Ensures consistent experiment setup by defining a global random seed and importing key utilities for model training, evaluation, and hyperparameter tuning.

### [2. Data Preparation](#prep)
Loads the preprocessed dataset (`df_preprocessed.csv`), displays its structure, separates features and target labels, and prepares data for modeling.

### [3. Random Forest](#random-forest)
Implements multiple Random Forest configurations for binary and multiclass classification, comparing fixed and randomized hyperparameter setups.

- [3.1 Binary RF (Fixed)](#31-binary-rf-fixed) – Trains a fixed-parameter Random Forest for binary classification and reports key metrics (ROC-AUC, PR-AUC, F1).  
- [3.2 Multiclass RF (Fixed)](#32-multiclass-rf-fixed) – Evaluates model performance across multiple disease classes with predefined parameters.  
- [3.3 Binary RF (Randomized Search)](#33-binary-rf-randomized-search) – Tunes hyperparameters using randomized search with PR-AUC optimization.  
- [3.4 Multiclass RF (Randomized Search)](#34-multiclass-rf-randomized-search) – Explores optimal hyperparameters using ROC-AUC for multiclass evaluation.

### [4. XGBoost](#xgboost)
Uses the XGBoost algorithm with early stopping and randomized search to achieve high-performance gradient-boosted models.

- [4.1 Binary XGBoost](#41-binary-xgboost) – Trains with early stopping, optimizing for precision-recall (PR-AUC) and balanced accuracy.  
- [4.2 Multiclass XGBoost](#42-multiclass-xgboost) – Applies the same approach to multiclass classification, comparing class-wise and macro-averaged metrics.

### [5. AdaBoost](#adaboost)
Tests the AdaBoost ensemble algorithm on both binary and multiclass versions of the dataset, using fixed and tuned parameter settings.

- [5.1 Binary Fixed](#51-binary-fixed) – Trains AdaBoost with fixed parameters on binary target.  
- [5.2 Multiclass Fixed](#52-multiclass-fixed) – Evaluates fixed AdaBoost setup on multiclass target.  
- [5.3 Binary Randomized Search](#53-binary-randomized-search) – Tunes AdaBoost parameters for binary classification with average precision scoring.  
- [5.4 Multiclass Randomized Search](#54-multiclass-randomized-search) – Conducts randomized hyperparameter search for multiclass setup and reports evaluation metrics.

*Each section includes detailed metrics (ROC-AUC, PR-AUC, Brier Score, Balanced Accuracy, F1-macro), confusion matrices, and classification reports for both binary and multiclass problems.*


<a id="imports"></a>
## Reproducibility & Imports  
---
This section provides consistent and organized experiment setup.  
- **Reproducibility:** Defines a fixed random seed (`SEED = 42`) for consistent results across runs.  
- **Path setup:** Adds project-level directories (`..` and `../scripts`) to the Python path so custom modules (e.g., `training`, `models`) can be imported.  
- **Imports:** Loads essential components for model training, tuning, and evaluation:
  - `get_cv`, `randomized_search`, `fit_eval`, `print_eval_report` – utilities for cross-validation, hyperparameter search, and performance reporting.
  - `make_rf_fixed`, `make_rf_base`, `rf_search_space`, `make_xgb_pipeline`, `xgb_search_space`, `make_ada_pipeline`, `ada_search_space` – functions for building pipelines and defining search spaces for Random Forest, XGBoost, and AdaBoost.
  - `train_test_split` from `sklearn.model_selection` – used for dataset splitting.
  - `xgboost` library and `clone` utility – for model replication and consistency checks.



In [ ]:
# Reproducibility
import os, sys
import pandas as pd

# Imports
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))

# Models & tools
from scripts.training import get_cv, randomized_search, fit_eval, print_eval_report
from scripts.models import make_rf_fixed, make_rf_base, rf_search_space, make_xgb_pipeline, xgb_search_space, make_ada_pipeline, ada_search_space
from scripts.models.random_forest import make_rf_fixed, make_rf_base, rf_search_space
from sklearn.model_selection import train_test_split
from sklearn.base import clone

SEED = 42

In [2]:
df = pd.read_csv("../data/df_preprocessed.csv")
df.head()

,age,sex,trestbps,chol,fbs,thalch,exang,oldpeak,num,dataset_Cleveland,...,cp_non-anginal,cp_typical angina,restecg_Missing,restecg_lv hypertrophy,restecg_normal,restecg_st-t abnormality,slope_Missing,slope_downsloping,slope_flat,slope_upsloping
0,1.007386,1,0.755639,-0.215021,1,0.474017,0,1.0000,0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1.432034,1,1.621952,0.790195,0,-1.153349,1,0.5000,2,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.432034,1,-0.688217,-0.290886,0,-0.339666,1,1.1875,1,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-1.752828,1,-0.110675,0.107407,0,1.907648,0,1.7500,0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,-1.328180,0,-0.110675,-0.765044,0,1.326446,0,0.4375,0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


### Data split

In [3]:
TARGET_COL = "num"
X = df.drop(columns=[TARGET_COL])
y_raw = df[TARGET_COL].astype(int)

<a id="random-forest"></a>
---
## Random Forest

### Binary RF (Fixed)

In [4]:
# Binary setup
y_bin = (y_raw > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_bin, test_size=0.2, random_state=SEED, stratify=y_bin
)

# Model
rf = make_rf_fixed("passthrough", seed=SEED)
rf.fit(X_tr, y_tr)
proba = rf.predict_proba(X_te)

# Evaluation
cls = rf.named_steps["clf"].classes_
prob_m, hard_m = print_eval_report(y_te, proba, classes=cls)


=== Probability metrics ===
ROC-AUC: 0.929
PR-AUC: 0.943
Brier: 0.111
BalancedAcc@0.5: 0.825

=== Hard-pred metrics ===
Accuracy: 0.832
BalancedAccuracy: 0.825
F1-macro: 0.828

Confusion matrix:
[[63 19]
 [12 90]]

Report:
              precision    recall  f1-score   support

           0      0.840     0.768     0.803        82
           1      0.826     0.882     0.853       102

    accuracy                          0.832       184
   macro avg      0.833     0.825     0.828       184
weighted avg      0.832     0.832     0.831       184



### Multiclass RF (Fixed)

In [5]:
# Multiclass setup
y_mc = y_raw.copy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_mc, test_size=0.2, random_state=SEED, stratify=y_mc
)

# Model
rf = make_rf_fixed("passthrough", seed=SEED)
rf.fit(X_tr, y_tr)
proba = rf.predict_proba(X_te)

# Evaluation
cls = rf.named_steps["clf"].classes_
prob_m, hard_m = print_eval_report(y_te, proba, classes=cls)


=== Probability metrics ===
ROC-AUC(macro-ovr): 0.863
PR-AUC(macro-ovr): 0.518
Brier: 0.093
BalancedAcc: 0.396

=== Hard-pred metrics ===
Accuracy: 0.625
BalancedAccuracy: 0.396
F1-macro: 0.392

Confusion matrix:
[[70 10  2  0  0]
 [13 35  3  2  0]
 [ 4  8  5  5  0]
 [ 3  8  4  5  1]
 [ 0  0  2  4  0]]

Report:
              precision    recall  f1-score   support

           0      0.778     0.854     0.814        82
           1      0.574     0.660     0.614        53
           2      0.312     0.227     0.263        22
           3      0.312     0.238     0.270        21
           4      0.000     0.000     0.000         6

    accuracy                          0.625       184
   macro avg      0.395     0.396     0.392       184
weighted avg      0.585     0.625     0.602       184



### Binary RF (Randomized Search)

In [6]:
#  Binary setup
y_bin = (y_raw > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_bin, test_size=0.2, random_state=SEED, stratify=y_bin
)

# Base model for search
base = make_rf_base("passthrough", seed=SEED)
cv = get_cv(5, seed=SEED)

# Randomized Search (PR-AUC is the best binary scorer)
search = randomized_search(
    model=base,
    param_space=rf_search_space(),
    X=X_tr, y=y_tr,
    cv=cv, n_iter=25,
    seed=SEED,
    scoring="average_precision",  # PR-AUC binary
)

# Best model & evaluation
best = search.best_estimator_
proba = best.predict_proba(X_te)
cls = best.named_steps["clf"].classes_

prob_m, hard_m = print_eval_report(y_te, proba, classes=cls)


=== Probability metrics ===
ROC-AUC: 0.934
PR-AUC: 0.949
Brier: 0.113
BalancedAcc@0.5: 0.831

=== Hard-pred metrics ===
Accuracy: 0.837
BalancedAccuracy: 0.831
F1-macro: 0.834

Confusion matrix:
[[64 18]
 [12 90]]

Report:
              precision    recall  f1-score   support

           0      0.842     0.780     0.810        82
           1      0.833     0.882     0.857       102

    accuracy                          0.837       184
   macro avg      0.838     0.831     0.834       184
weighted avg      0.837     0.837     0.836       184



### Multiclass RF (Randomized Search)

In [7]:
# Multiclass setup
y_mc = y_raw.copy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_mc, test_size=0.2, random_state=SEED, stratify=y_mc
)

# Base model for search
base = make_rf_base("passthrough", seed=SEED)
cv = get_cv(5, seed=SEED)

# Randomized Search
search = randomized_search(
    model=base,
    param_space=rf_search_space(),
    X=X_tr, y=y_tr,
    cv=cv, n_iter=25,
    seed=SEED,
    scoring="roc_auc_ovr",  # robust multiclass scorer
)

# Best model & evaluation
best = search.best_estimator_
proba = best.predict_proba(X_te)
cls = best.named_steps["clf"].classes_

prob_m, hard_m = print_eval_report(y_te, proba, classes=cls)


=== Probability metrics ===
ROC-AUC(macro-ovr): 0.858
PR-AUC(macro-ovr): 0.508
Brier: 0.094
BalancedAcc: 0.331

=== Hard-pred metrics ===
Accuracy: 0.603
BalancedAccuracy: 0.331
F1-macro: 0.303

Confusion matrix:
[[74  8  0  0  0]
 [15 35  2  1  0]
 [ 4 15  1  2  0]
 [ 4 16  0  1  0]
 [ 0  3  1  2  0]]

Report:
              precision    recall  f1-score   support

           0      0.763     0.902     0.827        82
           1      0.455     0.660     0.538        53
           2      0.250     0.045     0.077        22
           3      0.167     0.048     0.074        21
           4      0.000     0.000     0.000         6

    accuracy                          0.603       184
   macro avg      0.327     0.331     0.303       184
weighted avg      0.520     0.603     0.541       184



<a id="xgboost"></a>
## XGBoost 
*Model Training with Randomized Hyperparameter Search and Early Stopping*


### Binary XGBoost

In [8]:
# Binary target
y_bin = (y_raw > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_bin, test_size=0.2, random_state=SEED, stratify=y_bin
)

# Hold-out validation *inside training* for early stopping
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=y_tr
)

# Base pipeline
xgb_model = make_xgb_pipeline("passthrough", y_tr2, seed=SEED, task="binary")
xgb_model.set_params(clf__early_stopping_rounds=50)

# Randomized search
cv = get_cv(5, seed=SEED)
search = randomized_search(
    model=xgb_model,
    param_space=xgb_search_space(task="binary"),
    X=X_tr2, y=y_tr2,
    cv=cv, n_iter=30, seed=SEED,
    scoring="average_precision",
    fit_params={
        "clf__eval_set": [(X_val, y_val)],
        "clf__verbose": False,
    },
)

best = search.best_estimator_

# (Optional) Now refit on X_tr (train+val) with fresh small val split for stability
# Here we re-use the same (X_val, y_val) to keep it simple
best.fit(
    X_tr, y_tr,
    clf__eval_set=[(X_val, y_val)],
    clf__verbose=False,
)

proba = best.predict_proba(X_te)
cls = best.named_steps["clf"].classes_
prob_m, hard_m = print_eval_report(y_te, proba, classes=cls)

=== Probability metrics ===
ROC-AUC: 0.905
PR-AUC: 0.913
Brier: 0.119
BalancedAcc@0.5: 0.835

=== Hard-pred metrics ===
Accuracy: 0.842
BalancedAccuracy: 0.835
F1-macro: 0.838

Confusion matrix:
[[63 19]
 [10 92]]

Report:
              precision    recall  f1-score   support

           0      0.863     0.768     0.813        82
           1      0.829     0.902     0.864       102

    accuracy                          0.842       184
   macro avg      0.846     0.835     0.838       184
weighted avg      0.844     0.842     0.841       184



### Multiclass XGBoost

In [9]:
# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_mc, test_size=0.2, random_state=SEED, stratify=y_mc
)

# Hold-out validation for early stopping
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=y_tr
)

# Base pipeline
xgb_model = make_xgb_pipeline("passthrough", y_tr2, seed=SEED, task="multiclass")
xgb_model.set_params(clf__early_stopping_rounds=50)

# Randomized search
cv = get_cv(5, seed=SEED)
search = randomized_search(
    model=xgb_model,
    param_space=xgb_search_space(task="multiclass"),
    X=X_tr2, y=y_tr2,
    cv=cv, n_iter=30, seed=SEED,
    scoring="roc_auc_ovr",
    fit_params={
        "clf__eval_set": [(X_val, y_val)],
        "clf__verbose": False,
    },
)

best = search.best_estimator_

# Refit on X_tr with early stopping against the same small val set
best.fit(
    X_tr, y_tr,
    clf__eval_set=[(X_val, y_val)],
    clf__verbose=False,
)

proba = best.predict_proba(X_te)
cls = best.named_steps["clf"].classes_
prob_m, hard_m = print_eval_report(y_te, proba, classes=cls)

=== Probability metrics ===
ROC-AUC(macro-ovr): 0.823
PR-AUC(macro-ovr): 0.466
Brier: 0.107
BalancedAcc: 0.418

=== Hard-pred metrics ===
Accuracy: 0.614
BalancedAccuracy: 0.418
F1-macro: 0.417

Confusion matrix:
[[68  9  5  0  0]
 [14 30  5  2  2]
 [ 2  5  8  7  0]
 [ 4  2  6  7  2]
 [ 0  1  2  3  0]]

Report:
              precision    recall  f1-score   support

           0      0.773     0.829     0.800        82
           1      0.638     0.566     0.600        53
           2      0.308     0.364     0.333        22
           3      0.368     0.333     0.350        21
           4      0.000     0.000     0.000         6

    accuracy                          0.614       184
   macro avg      0.417     0.418     0.417       184
weighted avg      0.607     0.614     0.609       184



<a id="adaboost"></a>
## AdaBoost

### Binary Fixed

In [10]:
# Train/test split
y_bin = (y_raw > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_bin, test_size=0.2, random_state=SEED, stratify=y_bin
)

# Pipeline
ada_bin_fixed = make_ada_pipeline("passthrough", SEED)
ada_bin_fixed.fit(X_tr, y_tr)

# Probabilities
proba = ada_bin_fixed.predict_proba(X_te)

# Class order for alignment
try:
    cls = ada_bin_fixed.named_steps["clf"].classes_
except Exception:
    cls = None

# Evaluation
prob_m_bin_fixed, hard_m_bin_fixed = print_eval_report(y_te, proba, classes=cls)


=== Probability metrics ===
ROC-AUC: 0.928
PR-AUC: 0.942
Brier: 0.152
BalancedAcc@0.5: 0.850

=== Hard-pred metrics ===
Accuracy: 0.853
BalancedAccuracy: 0.850
F1-macro: 0.851

Confusion matrix:
[[67 15]
 [12 90]]

Report:
              precision    recall  f1-score   support

           0      0.848     0.817     0.832        82
           1      0.857     0.882     0.870       102

    accuracy                          0.853       184
   macro avg      0.853     0.850     0.851       184
weighted avg      0.853     0.853     0.853       184



### Multiclass Fixed

In [11]:
# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_mc, test_size=0.2, random_state=SEED, stratify=y_mc
)

# Pipeline (fixed)
ada_mc_fixed = make_ada_pipeline("passthrough", SEED)
ada_mc_fixed.fit(X_tr, y_tr)

# Probabilities
proba = ada_mc_fixed.predict_proba(X_te)

# Class order
try:
    cls = ada_mc_fixed.named_steps["clf"].classes_
except Exception:
    cls = None

# Evaluation
prob_m_mc_fixed, hard_m_mc_fixed = print_eval_report(y_te, proba, classes=cls)


=== Probability metrics ===
ROC-AUC(macro-ovr): 0.792
PR-AUC(macro-ovr): 0.404
Brier: 0.156
BalancedAcc: 0.301

=== Hard-pred metrics ===
Accuracy: 0.576
BalancedAccuracy: 0.301
F1-macro: 0.255

Confusion matrix:
[[74  8  0  0  0]
 [21 32  0  0  0]
 [ 5 17  0  0  0]
 [ 6 15  0  0  0]
 [ 0  6  0  0  0]]

Report:
              precision    recall  f1-score   support

           0      0.698     0.902     0.787        82
           1      0.410     0.604     0.489        53
           2      0.000     0.000     0.000        22
           3      0.000     0.000     0.000        21
           4      0.000     0.000     0.000         6

    accuracy                          0.576       184
   macro avg      0.222     0.301     0.255       184
weighted avg      0.429     0.576     0.492       184



### Binary Random

In [12]:
# Train/test split
y_bin = (y_raw > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_bin, test_size=0.2, random_state=SEED, stratify=y_bin
)

# Base model
base_bin = make_ada_pipeline("passthrough", SEED)

# CV + param space
cv = get_cv(5, seed=SEED)
param_space = ada_search_space()

# Randomized search
search_bin = randomized_search(
    model=base_bin,
    param_space=param_space,
    X=X_tr, y=y_tr,
    cv=cv, n_iter=30, seed=SEED,
    scoring="average_precision",
)

# Evaluation
ada_bin_best = search_bin.best_estimator_
proba = ada_bin_best.predict_proba(X_te)
cls = getattr(ada_bin_best.named_steps["clf"], "classes_", None)
print_eval_report(y_te, proba, classes=cls)


=== Probability metrics ===
ROC-AUC: 0.925
PR-AUC: 0.937
Brier: 0.158
BalancedAcc@0.5: 0.844

=== Hard-pred metrics ===
Accuracy: 0.848
BalancedAccuracy: 0.844
F1-macro: 0.845

Confusion matrix:
[[66 16]
 [12 90]]

Report:
              precision    recall  f1-score   support

           0      0.846     0.805     0.825        82
           1      0.849     0.882     0.865       102

    accuracy                          0.848       184
   macro avg      0.848     0.844     0.845       184
weighted avg      0.848     0.848     0.847       184



({'ROC-AUC': 0.9253945480631277,
  'PR-AUC': 0.9366952763107529,
  'Brier': 0.1580605792900342,
  'BalancedAcc@0.5': 0.8436154949784792},
 {'Accuracy': 0.8478260869565217,
  'BalancedAccuracy': 0.8436154949784792,
  'F1-macro': 0.8451923076923077})

### Multiclass Random

In [13]:
# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_mc, test_size=0.2, random_state=SEED, stratify=y_mc
)

# Base model
base_mc = make_ada_pipeline(pre="passthrough", seed=SEED)

# CV + param space
cv = get_cv(5, seed=SEED)
param_space = ada_search_space()

# Randomized search
search_mc = randomized_search(
    model=base_mc,
    param_space=param_space,
    X=X_tr, y=y_tr,
    cv=cv, n_iter=30, seed=SEED,
    scoring="roc_auc_ovr",
)

# Evaluation
ada_mc_best = search_mc.best_estimator_
proba = ada_mc_best.predict_proba(X_te)
cls = getattr(ada_mc_best.named_steps["clf"], "classes_", None)
print_eval_report(y_te, proba, classes=cls)

=== Probability metrics ===
ROC-AUC(macro-ovr): 0.834
PR-AUC(macro-ovr): 0.468
Brier: 0.152
BalancedAcc: 0.325

=== Hard-pred metrics ===
Accuracy: 0.592
BalancedAccuracy: 0.325
F1-macro: 0.293

Confusion matrix:
[[73  9  0  0  0]
 [16 34  3  0  0]
 [ 5 15  2  0  0]
 [ 6 10  5  0  0]
 [ 0  3  3  0  0]]

Report:
              precision    recall  f1-score   support

           0      0.730     0.890     0.802        82
           1      0.479     0.642     0.548        53
           2      0.154     0.091     0.114        22
           3      0.000     0.000     0.000        21
           4      0.000     0.000     0.000         6

    accuracy                          0.592       184
   macro avg      0.273     0.325     0.293       184
weighted avg      0.482     0.592     0.529       184



({'ROC-AUC(macro-ovr)': 0.833531193762826,
  'PR-AUC(macro-ovr)': 0.46781135735579954,
  'Brier': 0.15239613620366363,
  'BalancedAcc': 0.32453248546207586},
 {'Accuracy': 0.592391304347826,
  'BalancedAccuracy': 0.32453248546207586,
  'F1-macro': 0.292974122651542})